### 계획

+ 전처리된 데이터를 불러오기
+ 이상치를 만드는 함수 만들기
    + 급증, 급감, 정지, 노이즈
+ 정상 데이터의 무작위 위치에 이 함수들을 적용해서 이상치 주입
+ 이상치를 찾는 라벨을 컬럼으로 남기고 저장

In [1]:
# 라이브러리

import pandas as pd
import numpy as np

In [3]:
# 전처리 완료 데이터 불러오기

df = pd.read_csv("../data/processed/power_consumption_cleaned.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

print(df.shape)
df.head()

(2075259, 13)


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime,is_missing,missing_group,missing_block_size,short_missing,long_missing
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,False,1,6839,False,False
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,False,1,6839,False,False
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,False,1,6839,False,False
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,False,1,6839,False,False
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,False,1,6839,False,False


## 이상치 만들기

In [4]:
# 급증

def inject_spike(df, col, idx, magnitude=5):
    """
    지정한 위치(idx)의 값을 평소보다 magnitude배 크게 튀게 만듦
    
    Args:
        df: 데이터프레임
        col: 이상치를 넣을 컬럼명 (예: "Global_active_power")
        idx: 이상치를 넣을 위치(인덱스)
        magnitude: 원래 값의 몇 배로 튀게 할지 (기본 5배)
    """
    df = df.copy()
    original_value = df.loc[idx, col]
    df.loc[idx, col] = original_value * magnitude
    return df

In [5]:
# 급감

def inject_drop(df, col, idx, ratio=0.1):
    """
    지정한 위치(idx)의 값을 평소의 ratio배로 뚝 떨어뜨림
    
    Args:
        ratio: 원래 값의 몇 %로 떨어뜨릴지 (기본 0.1 = 10%)
    """
    df = df.copy()
    original_value = df.loc[idx, col]
    df.loc[idx, col] = original_value * ratio
    return df

In [7]:
# 정지

def inject_flatline(df, col, start_idx, duration=30):
    """
    start_idx부터 duration(분)만큼, 값을 하나로 고정시킴 (계량기 멈춤 흉내)
    
    Args:
        start_idx: 이상치 시작 위치
        duration: 몇 분 동안 고정시킬지
    """
    df = df.copy()
    fixed_value = df.loc[start_idx, col]  # 시작 시점 값으로 고정
    end_idx = start_idx + duration
    df.loc[start_idx:end_idx, col] = fixed_value
    return df

In [9]:
# 노이즈

def inject_noise(df, col, start_idx, duration=30, noise_std=2.0):
    """
    start_idx부터 duration(분)만큼, 랜덤한 노이즈를 더해서 값을 들쭉날쭉하게 만듦
    
    Args:
        noise_std: 노이즈의 크기(표준편차) - 클수록 더 심하게 들쭉날쭉해짐
    """
    df = df.copy()
    end_idx = start_idx + duration
    noise = np.random.normal(loc=0, scale=noise_std, size=(end_idx - start_idx + 1))
    df.loc[start_idx:end_idx, col] = df.loc[start_idx:end_idx, col].values + noise
    return df

## 이상치 갯수, 위치 정하기

In [11]:
np.random.seed(42)  # 결과 재현을 위해 시드 고정 (매번 같은 랜덤 결과 나오게)

def select_random_indices(df, n, min_gap=100, exclude_mask=None):
    """
    이상치를 넣을 위치를 무작위로 n개 선택
    
    Args:
        df: 데이터프레임
        n: 몇 개의 위치를 뽑을지
        min_gap: 선택된 위치들 사이 최소 간격(분) - 너무 몰리지 않게
        exclude_mask: 이미 결측(long_missing)인 구간은 제외하기 위한 마스크
    """
    valid_range = np.arange(len(df))
    if exclude_mask is not None:
        valid_range = valid_range[~exclude_mask.values]
    
    selected = []
    attempts = 0
    while len(selected) < n and attempts < n * 20:
        candidate = np.random.choice(valid_range)
        if all(abs(candidate - s) >= min_gap for s in selected):
            selected.append(candidate)
        attempts += 1
    
    return selected

In [12]:
# 결측(long_missing)이 아닌 구간에서만 위치 선택
n_spike = 200      # 급증 200개
n_drop = 200       # 급감 200개
n_flatline = 50    # 정지 50개 (구간형이라 개수를 적게)
n_noise = 50       # 노이즈 50개 (구간형이라 개수를 적게)

spike_indices = select_random_indices(df, n_spike, exclude_mask=df["long_missing"])
drop_indices = select_random_indices(df, n_drop, exclude_mask=df["long_missing"])
flatline_indices = select_random_indices(df, n_flatline, exclude_mask=df["long_missing"])
noise_indices = select_random_indices(df, n_noise, exclude_mask=df["long_missing"])

print(f"급증 위치 {len(spike_indices)}개, 급감 위치 {len(drop_indices)}개")
print(f"정지 위치 {len(flatline_indices)}개, 노이즈 위치 {len(noise_indices)}개")

급증 위치 200개, 급감 위치 200개
정지 위치 50개, 노이즈 위치 50개


## 이상치 넣고 라벨 만들기

In [13]:
# 이상치를 주입할 새 데이터프레임 (원본 보존을 위해 복사)
df_anomaly = df.copy()

# 라벨 컬럼 초기화 (기본값: 이상치 아님 = False, 유형은 "normal")
df_anomaly["is_anomaly"] = False
df_anomaly["anomaly_type"] = "normal"

target_col = "Global_active_power"

In [14]:
# 급증 주입

for idx in spike_indices:
    magnitude = np.random.uniform(3, 8)  # 3~8배 사이 랜덤하게 튀게
    original_value = df_anomaly.loc[idx, target_col]
    df_anomaly.loc[idx, target_col] = original_value * magnitude
    df_anomaly.loc[idx, "is_anomaly"] = True
    df_anomaly.loc[idx, "anomaly_type"] = "spike"

In [20]:
# 급감 주입

for idx in drop_indices:
    ratio = np.random.uniform(0.01, 0.2)  # 원래 값의 1~20% 수준으로 급감
    original_value = df_anomaly.loc[idx, target_col]
    df_anomaly.loc[idx, target_col] = original_value * ratio
    df_anomaly.loc[idx, "is_anomaly"] = True
    df_anomaly.loc[idx, "anomaly_type"] = "drop"

In [19]:
# 정지 주입

for start_idx in flatline_indices:
    duration = np.random.randint(15, 60)  # 15~60분 사이 랜덤 지속시간
    end_idx = min(start_idx + duration, len(df_anomaly) - 1)
    fixed_value = df_anomaly.loc[start_idx, target_col]
    
    df_anomaly.loc[start_idx:end_idx, target_col] = fixed_value
    df_anomaly.loc[start_idx:end_idx, "is_anomaly"] = True
    df_anomaly.loc[start_idx:end_idx, "anomaly_type"] = "flatline"

In [18]:
# 노이즈 주입

for start_idx in noise_indices:
    duration = np.random.randint(15, 60)
    end_idx = min(start_idx + duration, len(df_anomaly) - 1)
    noise_std = df_anomaly[target_col].std() * 1.5  # 전체 데이터 표준편차의 1.5배 크기 노이즈
    
    noise = np.random.normal(loc=0, scale=noise_std, size=(end_idx - start_idx + 1))
    df_anomaly.loc[start_idx:end_idx, target_col] = (
        df_anomaly.loc[start_idx:end_idx, target_col].values + noise
    )
    df_anomaly.loc[start_idx:end_idx, "is_anomaly"] = True
    df_anomaly.loc[start_idx:end_idx, "anomaly_type"] = "noise"

In [21]:
print("전체 이상치 개수:", df_anomaly["is_anomaly"].sum())
print()
print(df_anomaly["anomaly_type"].value_counts())

전체 이상치 개수: 4446

anomaly_type
normal      2070813
flatline       2220
noise          1826
spike           200
drop            200
Name: count, dtype: int64


In [22]:
# 저장
df_anomaly.to_csv("../data/processed/power_consumption_with_anomalies.csv", index=False)
print("저장 완료:", df_anomaly.shape)

저장 완료: (2075259, 15)
